# Gravity of Simple Bodies: Forward Models and Inversion

**Near-Surface Geophysics — Missouri S&T**

Before computers, gravity anomalies were interpreted with a handful of
shapes whose fields can be written down exactly: a **point mass**, a
**sphere**, a **cylinder**, a **prism**. They are still the first thing to
try, because they turn a curve of numbers into a depth and a mass in a few
lines of algebra. This notebook builds five of them,

| | Body | Infinite along strike? |
|---|---|---|
| 2-D | point mass (a *line* mass) | yes |
| 2-D | rectangular prism | yes |
| 3-D | point mass | no |
| 3-D | sphere | no |
| 3-D | vertical cylinder | no |

and uses them three ways:

1. **Forward models** (sections 2–4): the mathematics of each body and the
   anomaly it makes. The five bodies are chosen so that **their anomalies
   are the same, or nearly**: same peak, same width.
2. **Parametric inversion** (sections 5–6): add realistic noise, then fit
   each data set with the *right* shape and recover its depth, size and
   mass, with uncertainties. Then fit it with the *wrong* shape.
3. **Gridded inversion** (section 7): forget the shapes, divide the ground
   into cells, and solve for the density of every cell — the standard
   smallest-and-smoothest inversion.

The thread running through all three is **non-uniqueness**: gravity data
fix some things about a body very well and others not at all, and which
is which is the most useful thing to know before going to the field.

**Units.** Distances in metres. Density *contrast* $\Delta\rho$ in g/cm³
(the body's density minus that of the ground around it). Mass in tonnes
(1 t = 1000 kg). Gravity in **mGal**: 1 mGal = 10⁻⁵ m/s², about a
millionth of the Earth's pull; 1 µGal = 0.001 mGal. A field gravimeter
such as a Scintrex CG-5 repeats to about 5 µGal on a good day.

**Sign.** $z$ is depth, positive **down**, and $g_z$ is the extra pull
**downward**, so a mass excess makes a positive anomaly. Every formula
here is linear in $\Delta\rho$: a void ($\Delta\rho<0$) makes the same
curve upside down.

The whole notebook runs in under a minute.

## 0. Setup

Run this cell first. It installs `shallowgeo` if it is missing (or too old
to have the 3-D gravity code) and imports what the notebook uses.

In [ ]:
# Checks for the code this notebook needs specifically: an older install
# imports fine and then lacks it, and since the version string does not
# change between builds, a plain `pip install` reports "already satisfied".
import importlib
import sys

PACKAGE = ("shallow-geophysics @ "
           "git+https://github.com/Maurer-GEMLab/shallow-geophysics.git")

def shallowgeo_ready():
    for name in [n for n in sys.modules
                 if n == "shallowgeo" or n.startswith("shallowgeo.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
    try:
        from shallowgeo.potentialfields import gravity_kernel_3d, vertical_cylinder  # noqa: F401
        return True
    except ImportError:
        return False

if not shallowgeo_ready():
    %pip install -q --no-cache-dir "$PACKAGE"
if not shallowgeo_ready():
    %pip install -q --force-reinstall --no-deps --no-cache-dir "$PACKAGE"
if not shallowgeo_ready():
    raise ImportError(
        "shallowgeo.potentialfields is still missing the 3-D gravity code after "
        "installing. In Colab: Runtime > Restart session, then run this cell again."
    )

import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
from scipy.integrate import trapezoid
from scipy.optimize import least_squares

import shallowgeo.potentialfields as pf
from shallowgeo.potentialfields import GRAVITATIONAL_CONSTANT as G

MGAL = 1e5          # m/s^2 -> mGal
TONNE = 1e3         # kg
KG_M3 = 1e3         # g/cm^3 -> kg/m^3

# One colour per body, the same everywhere in the notebook.
COLOR = {
    "line mass": "#2a78d6",
    "2-D prism": "#eb6834",
    "point mass": "#1baf7a",
    "sphere": "#eda100",
    "vertical cylinder": "#e87ba4",
}
DATA_COLOR = "#52514e"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e4e3df", "grid.linewidth": 0.6,
                     "lines.linewidth": 2})
pd.set_option("display.precision", 3)

## 1. The bodies and the survey

Every number that defines the problem is in the next cell. All five bodies
have the same density contrast, **+1.0 g/cm³** — a dense body in lighter
ground, such as an ore lens or a block of dense rock in sediment.

**2-D bodies** (one profile across them; they run on for ever at right
angles to it):

* a **line mass** of 50.3 t per metre at 10 m depth — the same mass per
  metre as a horizontal cylinder of radius 4 m;
* a **prism** 10 m wide and 5 m thick, from 7 to 12 m deep.

**3-D bodies** (a map of stations above them), all centred under the
middle of the map:

* a **point mass** of 1697 t at 13 m depth;
* a **sphere** of radius 7.4 m centred at 13 m depth — the same mass;
* a **vertical cylinder** of radius 6 m from 6.3 to 22 m depth.

These were chosen so that all five anomalies peak at about **0.065 mGal**
and fall to half that about **10 m** either side; section 4 shows how.

**The survey.** The 2-D profile runs 80 m across the strike of the 2-D
bodies with a station every 2 m; the 3-D map covers 80 × 80 m with a
station every 4 m (441 stations — a few days' work with a CG-5, so in
practice a map is coarser than a profile). The sensor is 0.2 m above the
ground. The noise is **5 µGal**, independent from station to station.

In [ ]:
DRHO = 1.0                      # density contrast of every body, g/cm^3
HEIGHT = 0.2                    # gravimeter sensor above the ground, m

# --- 2-D bodies (x across strike) ----------------------------------------
LINE = dict(x0=0.0, depth=10.0, radius=4.0)                  # line mass = cylinder of this radius
LINE["mass_per_length"] = DRHO * KG_M3 * np.pi * LINE["radius"] ** 2    # kg/m
PRISM = dict(x_left=-5.0, x_right=5.0, top=7.0, bottom=12.0)

# --- 3-D bodies ------------------------------------------------------------
SPHERE = dict(x0=0.0, y0=0.0, depth=13.0, radius=7.4)
POINT = dict(x0=0.0, y0=0.0, depth=13.0,
             mass=DRHO * KG_M3 * 4 / 3 * np.pi * SPHERE["radius"] ** 3)   # kg
CYLINDER = dict(x0=0.0, y0=0.0, radius=6.0, top=6.3, bottom=22.0)

# --- Survey ----------------------------------------------------------------
X_PROFILE = np.arange(-40, 40.01, 2.0)                 # 2-D profile stations
XY_MAP = np.arange(-40, 40.01, 4.0)
X_MAP, Y_MAP = np.meshgrid(XY_MAP, XY_MAP)             # 3-D map stations
NOISE = 0.005                                          # mGal (5 µGal)
SEED = 111

x_fine = np.linspace(-60, 60, 1201)                    # for smooth curves
print(f"line mass:  {LINE['mass_per_length'] / TONNE:.1f} t/m")
print(f"point mass: {POINT['mass'] / TONNE:.0f} t")
print(f"profile: {X_PROFILE.size} stations;  map: {X_MAP.size} stations")

In [ ]:
def draw_bodies_2d(ax, **kw):
    """Outline the 2-D bodies on a cross-section."""
    ax.add_patch(Circle((LINE["x0"], LINE["depth"]), LINE["radius"], fill=False,
                        ec=COLOR["line mass"], ls="--", lw=1.5, **kw))
    ax.plot(LINE["x0"], LINE["depth"], "o", ms=5, color=COLOR["line mass"])
    ax.add_patch(Rectangle((PRISM["x_left"], PRISM["top"]), PRISM["x_right"] - PRISM["x_left"],
                           PRISM["bottom"] - PRISM["top"], fill=False,
                           ec=COLOR["2-D prism"], lw=1.5, **kw))


def draw_bodies_3d(ax, **kw):
    """Outline the 3-D bodies on a vertical section through their centres."""
    ax.plot(POINT["x0"], POINT["depth"], "o", ms=5, color=COLOR["point mass"])
    ax.add_patch(Circle((SPHERE["x0"], SPHERE["depth"]), SPHERE["radius"], fill=False,
                        ec=COLOR["sphere"], lw=1.5, **kw))
    ax.add_patch(Rectangle((CYLINDER["x0"] - CYLINDER["radius"], CYLINDER["top"]),
                           2 * CYLINDER["radius"], CYLINDER["bottom"] - CYLINDER["top"],
                           fill=False, ec=COLOR["vertical cylinder"], lw=1.5, **kw))


fig, axs = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
draw_bodies_2d(axs[0])
axs[0].set(title="2-D bodies: cross-section (they run on into the page)")
axs[0].legend(handles=[
    plt.Line2D([], [], color=COLOR["line mass"], marker="o", ls="--", label="line mass (and its cylinder)"),
    plt.Line2D([], [], color=COLOR["2-D prism"], label="2-D prism")], loc="lower left")
draw_bodies_3d(axs[1])
axs[1].set(title="3-D bodies: vertical section through the centre")
axs[1].legend(handles=[
    plt.Line2D([], [], color=COLOR["point mass"], marker="o", ls="", label="point mass"),
    plt.Line2D([], [], color=COLOR["sphere"], label="sphere"),
    plt.Line2D([], [], color=COLOR["vertical cylinder"], label="vertical cylinder")], loc="lower left")
for ax in axs:
    ax.set(xlim=(-25, 25), ylim=(28, 0), aspect="equal", xlabel="x (m)")
    ax.axhline(0, color="k", lw=1)
axs[0].set_ylabel("depth (m)")
plt.tight_layout()

## 2. Forward models in 2-D

### 2.1 Newton's law, and the 2-D "point mass"

A point mass $M$ at distance $r$ pulls with $GM/r^2$ along the line
joining them. A gravimeter measures only the **vertical** component, which
is that pull times $z/r$:

$$g_z = \frac{G M z}{r^3}, \qquad r^2 = x^2 + y^2 + z^2,$$

with $G = 6.674\times10^{-11}$ m³ kg⁻¹ s⁻², $x, y$ the horizontal offsets
from the station to the mass and $z$ its depth below the station.

A **2-D body** runs on for ever along strike ($y$). The 2-D version of a
point is therefore a **line** of mass $\lambda$ per metre, and its field is
the point-mass field added up along the line:

$$g_z = G\lambda z\int_{-\infty}^{\infty}\frac{dy}{(x^2+y^2+z^2)^{3/2}}
      = \frac{2G\lambda\, z}{x^2 + z^2}.$$

Two things follow at once:

* the **peak**, over the line ($x=0$), is $2G\lambda/z$;
* the anomaly falls to **half** its peak where $x^2 = z^2$, so the
  **half-width** $x_{1/2}$ equals the depth: $\;z = x_{1/2}$.

A horizontal cylinder of radius $R$ makes exactly the same field outside
itself, with $\lambda = \Delta\rho\,\pi R^2$ — by symmetry, as for a
sphere in 3-D. So the line mass is also the **horizontal cylinder**, and
gravity alone cannot say which, or how wide the cylinder is.

(The depth is measured from the *sensor*, so `shallowgeo` adds the
0.2 m sensor height to every depth below.)

In [ ]:
def half_width(x, g):
    """Distance from the peak to where g first falls to half of it, averaged
    over both sides, by linear interpolation between samples."""
    k = np.argmax(g)
    half = g[k] / 2
    out = []
    for side in (slice(k, None), slice(k, None, -1)):
        xs, gs = x[side], g[side]
        j = np.argmax(gs < half)
        out.append(abs(np.interp(half, [gs[j], gs[j - 1]], [xs[j], xs[j - 1]]) - x[k]))
    return float(np.mean(out))


def line_mass_2d(x, x0, depth, lam):
    return pf.line_mass(x, x0=x0, depth=depth, mass_per_length=lam, height=HEIGHT)


g_line = line_mass_2d(x_fine, LINE["x0"], LINE["depth"], LINE["mass_per_length"])
z = LINE["depth"] + HEIGHT
print(f"peak: {g_line.max():.4f} mGal   (2 G lambda / z = "
      f"{2 * G * LINE['mass_per_length'] / z * MGAL:.4f})")
print(f"half-width: {half_width(x_fine, g_line):.2f} m   (depth below the sensor = {z:.2f} m)")

### 2.2 The 2-D rectangular prism

Any 2-D body is a bundle of line masses. Cut its cross-section into strips
$dx'\,dz'$, each a line mass $\lambda = \Delta\rho\,dx'\,dz'$, and add
them up:

$$g_z(x) = 2G\Delta\rho \int_{z_1}^{z_2}\!\int_{x_1}^{x_2}
          \frac{z'}{(x'-x)^2 + z'^2}\,dx'\,dz'.$$

For a rectangle the double integral has a closed form (Telford et al.,
1990; Blakely, 1995). With $X = x' - x$ measured from the station to a
corner, $Z$ the corner's depth below the station and
$F(X, Z) = X\ln\sqrt{X^2+Z^2} + Z\arctan(X/Z)$,

$$g_z = 2G\Delta\rho\,\big[F(X_2,Z_2) - F(X_1,Z_2) - F(X_2,Z_1) + F(X_1,Z_1)\big],$$

a signed sum over the four corners. This is the same formula
`pf.gravity_kernel` uses for every cell of a 2-D mesh in section 7.

A prism far away looks like a line mass of $\lambda = \Delta\rho\,w\,t$
(width × thickness) at its centre. This one, 10 m × 5 m, has
$\lambda$ = 50.0 t/m, almost the same as the line mass; its centre is
half a metre shallower to make up for being wide and flat.

In [ ]:
def prism_2d(x, x_center, width, top, thickness, drho=DRHO):
    return pf.prism_2d(x, x_left=x_center - width / 2, x_right=x_center + width / 2,
                       top=top, bottom=top + thickness, density=drho, height=HEIGHT)


PRISM_P = dict(x_center=(PRISM["x_left"] + PRISM["x_right"]) / 2,
               width=PRISM["x_right"] - PRISM["x_left"],
               top=PRISM["top"], thickness=PRISM["bottom"] - PRISM["top"])
g_prism = prism_2d(x_fine, **PRISM_P)
lam_prism = DRHO * KG_M3 * PRISM_P["width"] * PRISM_P["thickness"]
print(f"prism: lambda = {lam_prism / TONNE:.1f} t/m, peak {g_prism.max():.4f} mGal, "
      f"half-width {half_width(x_fine, g_prism):.2f} m")
print(f"largest difference from the line mass: "
      f"{np.abs(g_prism - g_line).max() * 1e3:.2f} µGal  (noise: {NOISE * 1e3:.0f} µGal)")

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(x_fine, g_line, color=COLOR["line mass"], label="line mass, 10 m deep")
ax.plot(x_fine, g_prism, color=COLOR["2-D prism"], ls="--", label="prism, 10 × 5 m, 7–12 m deep")
ax.set(xlabel="x (m)", ylabel="$g_z$ (mGal)", title="Two different 2-D bodies, one anomaly")
ax.legend()
plt.tight_layout()

## 3. Forward models in 3-D

### 3.1 The point mass

Back to Newton, now on a map of stations:

$$g_z(x, y) = \frac{G M z}{\big[(x-x_0)^2 + (y-y_0)^2 + z^2\big]^{3/2}}.$$

The anomaly is **circular**, peaks at $GM/z^2$ over the mass, and falls to
half where $(1 + x^2/z^2)^{3/2} = 2$, that is at

$$x_{1/2} = z\sqrt{2^{2/3}-1} \approx 0.766\,z
\qquad\Longrightarrow\qquad z \approx 1.305\,x_{1/2}.$$

Compare the 2-D rule $z = x_{1/2}$: **the same curve means a deeper body
if it is 3-D.** A profile on its own cannot tell you which rule to use;
a map can.

### 3.2 The sphere

Newton's **shell theorem**: outside a spherically symmetric body, its
field is that of a point mass at its centre, with

$$M = \tfrac43\pi R^3\Delta\rho.$$

So the sphere's anomaly is *identical* to the point mass's, not merely
similar. From the outside, gravity measures the **mass and the depth to
the centre** — not the radius, not the density contrast, and not the depth
to the top. A big sphere of small contrast and a small sphere of large
contrast are the same to a gravimeter.

In [ ]:
def point_mass_3d(x, y, x0, y0, depth, mass):
    return pf.point_mass(x, y, x0=x0, y0=y0, depth=depth, mass=mass, height=HEIGHT)


def sphere_3d(x, y, x0, y0, depth, radius, drho=DRHO):
    return pf.sphere(x, y, x0=x0, y0=y0, depth=depth, radius=radius, density=drho,
                     height=HEIGHT)


g_point = point_mass_3d(x_fine, 0.0, **POINT)          # profile along y = 0
g_sphere = sphere_3d(x_fine, 0.0, **SPHERE)
z = POINT["depth"] + HEIGHT
print(f"point mass: peak {g_point.max():.4f} mGal (G M / z^2 = {G * POINT['mass'] / z**2 * MGAL:.4f}), "
      f"half-width {half_width(x_fine, g_point):.2f} m (0.766 z = {0.766 * z:.2f})")
print(f"sphere minus point mass: largest difference {np.abs(g_sphere - g_point).max():.1e} mGal")

### 3.3 The vertical cylinder

A vertical cylinder — a pipe, a sinkhole fill, a plug — is a bundle of
**vertical line segments**. One segment of cross-section $dA$, at
horizontal distance $s$ from the station and running from depth $z_1$ to
$z_2$ below it, has mass $\Delta\rho\,dA$ per metre of length, and

$$dg_z = G\Delta\rho\,dA\int_{z_1}^{z_2}\frac{z\,dz}{(s^2+z^2)^{3/2}}
       = G\Delta\rho\,dA\left[\frac{1}{\sqrt{s^2+z_1^2}} - \frac{1}{\sqrt{s^2+z_2^2}}\right].$$

The depth integral is exact; what is left is a sum over the disc,

$$g_z = G\Delta\rho\int_0^{R}\!\!\int_0^{2\pi}
\left[\frac{1}{\sqrt{s^2+z_1^2}} - \frac{1}{\sqrt{s^2+z_2^2}}\right] r\,d\theta\,dr,$$

with $s$ the distance from the station to the point $(r, \theta)$ of the
disc. There is no closed form off the axis, so `pf.vertical_cylinder` does
it by quadrature: Gauss–Legendre in $r$, equally spaced points in
$\theta$. Directly over the axis there *is* a closed form (Telford et al.,
1990), a good check:

$$g_z(0) = 2\pi G\Delta\rho\left[L + \sqrt{z_1^2+R^2} - \sqrt{z_2^2+R^2}\right],
\qquad L = z_2 - z_1.$$

In [ ]:
def cylinder_3d(x, y, x0, y0, radius, top, bottom, drho=DRHO, n=24):
    return pf.vertical_cylinder(x, y, x0=x0, y0=y0, radius=radius, top=top, bottom=bottom,
                                density=drho, height=HEIGHT, n=n)


exact = pf.vertical_cylinder_on_axis(radius=CYLINDER["radius"], top=CYLINDER["top"],
                                     bottom=CYLINDER["bottom"], density=DRHO, height=HEIGHT)
for n in (2, 4, 8, 16, 24):
    q = cylinder_3d(0.0, 0.0, **CYLINDER, n=n)
    print(f"n = {n:2d}: on-axis {float(q):.7f} mGal, error {abs(float(q) - exact) * 1e3:.1e} µGal")
print(f"exact:   {exact:.7f} mGal")

g_cyl = cylinder_3d(x_fine, 0.0, **CYLINDER)
print(f"\ncylinder: peak {g_cyl.max():.4f} mGal, half-width {half_width(x_fine, g_cyl):.2f} m")

### 3.4 The three 3-D anomalies as maps

All three are circular — a 3-D body that is round in plan always makes a
round anomaly — and all three are, to the eye, the same map.

In [ ]:
maps = {
    "point mass": point_mass_3d(X_MAP, Y_MAP, **POINT),
    "sphere": sphere_3d(X_MAP, Y_MAP, **SPHERE),
    "vertical cylinder": cylinder_3d(X_MAP, Y_MAP, **CYLINDER),
}
vmax = max(m.max() for m in maps.values())
fig, axs = plt.subplots(1, 3, figsize=(13, 3.9), sharey=True)
for ax, (name, g) in zip(axs, maps.items()):
    pc = ax.pcolormesh(X_MAP, Y_MAP, g, cmap="Blues", vmin=0, vmax=vmax, shading="nearest")
    ax.contour(X_MAP, Y_MAP, g, levels=[vmax / 2], colors="k", linewidths=0.8)
    ax.set(title=name, aspect="equal", xlabel="x (m)")
    ax.grid(False)
axs[0].set_ylabel("y (m)")
fig.colorbar(pc, ax=axs, label="$g_z$ (mGal)", shrink=0.85)
fig.suptitle("3-D anomalies on the survey grid (black: half of the peak)", y=1.0)

## 4. Five bodies, one anomaly

How were the parameters chosen? Start from the 2-D line mass (10 m deep,
50.3 t/m) and match everything else to it:

* **2-D prism.** Keep $\lambda = \Delta\rho\,w\,t$ the same (10 × 5 m),
  and nudge the depth until the curves agree.
* **3-D point mass.** Match the half-width: $z_{3D} = 1.305\,x_{1/2} \approx 13$ m.
  Then match the peak: $GM/z_{3D}^2 = 2G\lambda/z_{2D}$ gives
  $M = 2\lambda z_{3D}^2/z_{2D} \approx 1700$ t.
* **Sphere.** Same centre and mass: $R = (3M/4\pi\Delta\rho)^{1/3} = 7.4$ m.
* **Vertical cylinder.** Pick a radius (6 m) and find the top and bottom
  that fit the sphere's anomaly best, by the same least squares used in
  section 6 (6.3 and 22 m).

The table measures how alike they are. The last column is the important
one: the **root-mean-square difference** from the reference curve of each
group, against a noise of 5 µGal.

In [ ]:
curves_2d = {"line mass": g_line, "2-D prism": g_prism}
curves_3d = {"point mass": g_point, "sphere": g_sphere, "vertical cylinder": g_cyl}
on_profile = np.abs(x_fine) <= 40
rows = []
for group, curves, ref in (("2-D", curves_2d, g_line), ("3-D", curves_3d, g_point)):
    for name, g in curves.items():
        rows.append({"": group, "body": name, "peak (mGal)": g.max(),
                     "half-width (m)": half_width(x_fine, g),
                     "at x = 40 m (mGal)": g[np.argmin(abs(x_fine - 40))],
                     "rms diff. from first of group (µGal)":
                         1e3 * np.sqrt(np.mean((g - ref)[on_profile] ** 2))})
rows.append({"": "2-D vs 3-D", "body": "line mass vs point mass",
             "rms diff. from first of group (µGal)":
                 1e3 * np.sqrt(np.mean((g_line - g_point)[on_profile] ** 2))})
pd.DataFrame(rows).set_index(["", "body"])

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(14, 6), height_ratios=[2, 1], sharex="col")
for col, (title, curves, ref) in enumerate((("2-D bodies", curves_2d, g_line),
                                            ("3-D bodies (profile through the centre)",
                                             curves_3d, g_point))):
    for i, (name, g) in enumerate(curves.items()):
        axs[0, col].plot(x_fine, g, color=COLOR[name], ls=["-", "--", ":"][i], label=name)
        if i:
            axs[1, col].plot(x_fine, (g - ref) * 1e3, color=COLOR[name], ls=["-", "--", ":"][i],
                             label=f"{name} − {next(iter(curves))}")
    axs[0, col].set(title=title, ylabel="$g_z$ (mGal)")
    axs[0, col].legend()
    axs[1, col].axhspan(-NOISE * 1e3, NOISE * 1e3, color="#e4e3df", label="± noise (5 µGal)")
    axs[1, col].set(xlabel="x (m)", ylabel="difference (µGal)", ylim=(-6, 6))
    axs[1, col].legend(fontsize=8, loc="lower right")

# 2-D against 3-D: same peak and width, different tails.
ax = axs[0, 2]
ax.plot(x_fine, g_line, color=COLOR["line mass"], label="2-D line mass")
ax.plot(x_fine, g_point, color=COLOR["point mass"], ls="--", label="3-D point mass")
ax.set(title="2-D against 3-D", ylabel="$g_z$ (mGal)")
ax.legend()
ax = axs[1, 2]
ax.semilogy(x_fine, g_line, color=COLOR["line mass"])
ax.semilogy(x_fine, g_point, color=COLOR["point mass"], ls="--")
ax.axhline(NOISE, color=DATA_COLOR, lw=1, ls=":")
ax.text(-58, NOISE * 1.15, "noise", fontsize=8, color=DATA_COLOR)
ax.set(xlabel="x (m)", ylabel="$g_z$ (mGal), log", ylim=(5e-4, 0.1))
plt.tight_layout()

Within each group the curves differ by **about a microgal at most**, and
by half that or less on average — several times less than the noise. No
survey could tell them apart.

Between the groups, the peak and the width agree but the **tails** do
not: far from a line mass $g_z \propto 1/x^2$, far from a point mass
$g_z \propto 1/x^3$. At 40 m the 2-D anomaly is twice the 3-D one — but
both are below the noise there, and the rms difference over the profile
is about 2 µGal. **A single profile barely distinguishes a 2-D body from
a 3-D one.** A map does, at a glance: one anomaly is elongated, the other
round.

## 5. Synthetic data

Now pretend these are field data: compute each body's anomaly at the
stations and add independent Gaussian noise of 5 µGal. Each data set gets
its own noise. Change `SEED` in section 1 for a different draw.

Two things to keep in mind:

* The data come from the **exact** formulas, not from the cells used in
  section 7, so the gridded inversion is not handed its own forward model
  (the "inverse crime").
* Real data also contain a **regional** trend, drift and terrain errors that
  are not white noise. Here the noise is the only thing standing between us
  and the truth, which makes this a best case.

In [ ]:
rng = np.random.default_rng(SEED)
clean = {
    "line mass": line_mass_2d(X_PROFILE, LINE["x0"], LINE["depth"], LINE["mass_per_length"]),
    "2-D prism": prism_2d(X_PROFILE, **PRISM_P),
    "point mass": maps["point mass"].ravel(),
    "sphere": maps["sphere"].ravel(),
    "vertical cylinder": maps["vertical cylinder"].ravel(),
}
data = {name: g + rng.normal(0, NOISE, g.shape) for name, g in clean.items()}
for name, d in data.items():
    print(f"{name:18s} {d.size:4d} data, peak {d.max():.3f} mGal, "
          f"peak / noise = {clean[name].max() / NOISE:.0f}")

fig, axs = plt.subplots(1, 2, figsize=(13, 3.8), width_ratios=[1.4, 1])
for name in ("line mass", "2-D prism"):
    axs[0].plot(x_fine, curves_2d[name], color=COLOR[name], lw=1.5, label=f"{name}, noise-free")
    axs[0].errorbar(X_PROFILE, data[name], NOISE, fmt="o", ms=4, color=COLOR[name],
                    ecolor=DATA_COLOR, elinewidth=0.8, label=f"{name}, observed")
axs[0].set(xlabel="x (m)", ylabel="$g_z$ (mGal)", title="2-D profiles")
axs[0].legend(fontsize=8)
pc = axs[1].pcolormesh(X_MAP, Y_MAP, data["vertical cylinder"].reshape(X_MAP.shape),
                       cmap="Blues", shading="nearest")
axs[1].set(aspect="equal", xlabel="x (m)", ylabel="y (m)",
           title="3-D map: the vertical cylinder, with noise")
axs[1].grid(False)
fig.colorbar(pc, ax=axs[1], label="$g_z$ (mGal)")
plt.tight_layout()

## 6. Parametric inversion: fit the right shape

Suppose we know — from geology, a borehole, or the shape of the anomaly —
**what kind of body** it is, and want its position, depth and size. The
unknowns are a handful of numbers $\mathbf p$ (for the point mass:
$x_0, y_0, z, M$), and the forward model $g(\mathbf p)$ is one of the
formulas above. It is **non-linear** in most of them (depth appears in a
denominator), so there is no one-step answer.

### 6.1 Non-linear least squares

Find the $\mathbf p$ that minimises the misfit

$$\chi^2(\mathbf p) = \sum_{i=1}^{N}\left(\frac{d_i - g_i(\mathbf p)}{\sigma_i}\right)^2 .$$

Starting from a guess, linearise about the current $\mathbf p$ with the
**Jacobian** $J_{ij} = \partial r_i/\partial p_j$ of the weighted residuals
$r_i = (g_i(\mathbf p) - d_i)/\sigma_i$, step to the minimum of the linear
problem, repeat. That is Gauss–Newton; the **Levenberg–Marquardt** method
in `scipy.optimize.least_squares` adds a damping term so it cannot step
wildly when the guess is poor.

When it has converged:

* $\chi^2/N \approx 1$ if the shape is right and the noise is what we said.
* The **covariance** of the parameters is approximately
  $\mathbf C = (\mathbf J^T\mathbf J)^{-1}$. Its diagonal gives a standard
  deviation $\sigma_p$ for each parameter; the **correlation**
  $C_{jk}/\sigma_j\sigma_k$ says which parameters trade off against each
  other. A correlation near ±1 means the data constrain only a
  combination of the two.
* If two parameters enter the formula only in a fixed combination,
  $\mathbf J^T\mathbf J$ is **singular**: its condition number is huge and
  no amount of data separates them.

### 6.2 A starting guess from the rules of thumb

Least squares needs a start. The half-width rules of sections 2 and 3 give
a good one straight from the data: the peak gives $x_0$ (and $y_0$), the
half-width gives the depth, and the peak value then gives the mass.

In [ ]:
def fit(forward, names, p0, d, std=NOISE, truth=None, bounds=(-np.inf, np.inf)):
    """Least-squares fit of forward(*p) to data d. Returns a dict with the
    parameters, their 1-sigma uncertainties, correlations and chi^2 / N."""
    sol = least_squares(lambda p: (forward(*p) - d) / std, p0, bounds=bounds,
                        x_scale="jac", xtol=1e-12, ftol=1e-12)
    JtJ = sol.jac.T @ sol.jac
    cond = np.linalg.cond(JtJ)
    C = np.linalg.inv(JtJ) if cond < 1e14 else np.full(JtJ.shape, np.nan)
    sd = np.sqrt(np.diag(C))
    table = pd.DataFrame({"start": p0, "fitted": sol.x, "± 1σ": sd}, index=names)
    if truth is not None:
        table.insert(0, "true", truth)
    return {"p": sol.x, "sd": sd, "table": table, "chi2": 2 * sol.cost / d.size,
            "corr": pd.DataFrame(C / np.outer(sd, sd), index=names, columns=names),
            "cond": cond, "predicted": forward(*sol.x), "nfev": sol.nfev}


def report(result, title):
    print(f"{title}:  chi^2/N = {result['chi2']:.2f},  "
          f"condition number of J^T J = {result['cond']:.1e}")
    return result["table"]


def guess_2d(x, d):
    k = np.argmax(d)
    z = half_width(x, d) - HEIGHT                       # 2-D rule: z = x_1/2
    lam = d[k] / MGAL * (z + HEIGHT) / (2 * G)
    return x[k], z, lam


def guess_3d(d_map):
    d_map = d_map.reshape(X_MAP.shape)
    iy, ix = np.unravel_index(np.argmax(d_map), d_map.shape)
    hw = np.mean([half_width(XY_MAP, d_map[iy, :]), half_width(XY_MAP, d_map[:, ix])])
    z = 1.305 * hw - HEIGHT                             # 3-D rule: z = 1.305 x_1/2
    mass = d_map[iy, ix] / MGAL * (z + HEIGHT) ** 2 / G
    return XY_MAP[ix], XY_MAP[iy], z, mass

### 6.3 2-D line mass

Three unknowns: position $x_0$, depth $z$, and mass per metre $\lambda$
(fitted in tonnes per metre, to keep the numbers near 1).

In [ ]:
x0, z0, lam0 = guess_2d(X_PROFILE, data["line mass"])
fit_line = fit(lambda x0, z, lam_t: line_mass_2d(X_PROFILE, x0, z, lam_t * TONNE),
               ["x0 (m)", "depth (m)", "λ (t/m)"], [x0, z0, lam0 / TONNE], data["line mass"],
               truth=[LINE["x0"], LINE["depth"], LINE["mass_per_length"] / TONNE])
report(fit_line, "line mass")

In [ ]:
fit_line["corr"]

Position, depth and mass come back within one or two standard deviations
of the truth — which is what $\pm1\sigma$ means: the truth should fall
outside it about a third of the time. The uncertainties are a few percent:
about 6 % on depth and 4 % on mass. Depth and mass are positively
correlated (a deeper line needs more mass to give the same peak), but not
so strongly that they cannot be separated, because depth also sets the
width.

What the fit **cannot** give is the radius of the cylinder: only
$\lambda = \Delta\rho\,\pi R^2$ is determined. With $\Delta\rho$ known
(here 1.0), $R = \sqrt{\lambda/\pi\Delta\rho}$ follows — but the
$\Delta\rho$ came from outside the gravity data.

### 6.4 2-D prism

Four unknowns with $\Delta\rho$ known: centre $x_c$, width $w$, depth to
top, thickness $t$. The start is a square prism with the line mass's
$\lambda$ at the line mass's depth.

In [ ]:
x0, z0, lam0 = guess_2d(X_PROFILE, data["2-D prism"])
side = np.sqrt(lam0 / (DRHO * KG_M3))
p0 = [x0, side, z0 - side / 2, side]
fit_prism = fit(lambda xc, w, top, t: prism_2d(X_PROFILE, xc, w, top, t),
                ["x_center (m)", "width (m)", "top (m)", "thickness (m)"], p0, data["2-D prism"],
                truth=list(PRISM_P.values()),
                bounds=([-50, 0.1, 0, 0.1], [50, 60, 40, 40]))
report(fit_prism, "2-D prism")

In [ ]:
fit_prism["corr"]

The fit is excellent ($\chi^2/N\approx1$) but look at the uncertainties:
width and thickness are uncertain by metres and correlated at −0.99; only
the centre and the top are well determined.
The data fix the **centre** and the **mass per metre** $\lambda = \Delta\rho\,w\,t$
well; how that mass is shared between width and thickness hardly changes
the anomaly (we saw in section 2.2 that even a line mass fits). Section 6.7
shows the spread directly.

### 6.5 3-D point mass

Four unknowns: $x_0, y_0$, depth, mass.

In [ ]:
p0 = guess_3d(data["point mass"])
fit_point = fit(lambda x0, y0, z, m_t: point_mass_3d(X_MAP, Y_MAP, x0, y0, z, m_t * TONNE).ravel(),
                ["x0 (m)", "y0 (m)", "depth (m)", "M (t)"], [*p0[:3], p0[3] / TONNE],
                data["point mass"],
                truth=[POINT["x0"], POINT["y0"], POINT["depth"], POINT["mass"] / TONNE])
report(fit_point, "point mass")

With 441 stations around it, the point mass is pinned down tightly —
depth to a few tenths of a metre and mass to about 3 %.

### 6.6 Sphere: what happens when a parameter cannot be resolved

Try the obvious parameterisation first: centre, radius **and** density
contrast — five unknowns. Run it from two different starting guesses.

In [ ]:
names = ["x0 (m)", "y0 (m)", "depth (m)", "R (m)", "Δρ (g/cm³)"]
fwd = lambda x0, y0, z, R, drho: sphere_3d(X_MAP, Y_MAP, x0, y0, z, R, drho).ravel()
bounds = ([-50, -50, 0.5, 0.5, 0.01], [50, 50, 60, 30, 5])
x0, y0, z0, m0 = guess_3d(data["sphere"])
for R0, drho0 in ((5.0, 2.0), (9.0, 0.5)):
    r = fit(fwd, names, [x0, y0, z0, R0, drho0], data["sphere"], bounds=bounds)
    R, drho = r["p"][3], r["p"][4]
    print(f"start R = {R0}, Δρ = {drho0}:  fitted R = {R:.2f} m, Δρ = {drho:.3f} g/cm³, "
          f"M = {4 / 3 * np.pi * R**3 * drho * KG_M3 / TONNE:.0f} t,  chi^2/N = {r['chi2']:.3f},  "
          f"cond(J^T J) = {r['cond']:.0e}")
print(f"truth:                      R = {SPHERE['radius']:.2f} m, Δρ = {DRHO:.3f} g/cm³, "
      f"M = {POINT['mass'] / TONNE:.0f} t")

Both fits are equally good, they disagree about $R$ and $\Delta\rho$, and
they agree about the mass. The radius and the density contrast enter only
as $R^3\Delta\rho$, so the Jacobian has two columns that are proportional
and $\mathbf J^T\mathbf J$ is singular (a condition number near 10¹⁶ is
"singular to machine precision"). The optimiser simply stops wherever it
first reaches the valley floor.

The cure is not more data but **less freedom**: fix one of them from
outside information. With $\Delta\rho$ = 1.0 g/cm³ known:

In [ ]:
fit_sphere = fit(lambda x0, y0, z, R: sphere_3d(X_MAP, Y_MAP, x0, y0, z, R).ravel(),
                 names[:4], [x0, y0, z0, (3 * m0 / (4 * np.pi * DRHO * KG_M3)) ** (1 / 3)],
                 data["sphere"], truth=list(SPHERE.values()),
                 bounds=([-50, -50, 0.5, 0.5], [50, 50, 60, 30]))
report(fit_sphere, "sphere, Δρ fixed")

The radius now looks very precise, ±0.1 m. That precision is borrowed: it
assumes $\Delta\rho$ is exactly 1.0. Since only $R^3\Delta\rho$ is
determined, a 10 % error in the assumed contrast moves $R$ by about 3 %,
0.25 m — three times the quoted uncertainty.

### 6.7 Vertical cylinder

Five unknowns with $\Delta\rho$ known: $x_0, y_0$, radius, top and bottom.
For a start, take a cylinder as wide as it is tall, centred on the
point-mass depth.

In [ ]:
x0, y0, z0, m0 = guess_3d(data["vertical cylinder"])
R0 = (m0 / (2 * np.pi * DRHO * KG_M3)) ** (1 / 3)        # height = 2R
fit_cyl = fit(lambda x0, y0, R, top, bottom:
              cylinder_3d(X_MAP, Y_MAP, x0, y0, R, top, bottom, n=12).ravel(),
              ["x0 (m)", "y0 (m)", "radius (m)", "top (m)", "bottom (m)"],
              [x0, y0, R0, max(z0 - R0, 0.5), z0 + R0], data["vertical cylinder"],
              truth=list(CYLINDER.values()),
              bounds=([-50, -50, 0.5, 0, 1], [50, 50, 40, 40, 200]))
report(fit_cyl, "vertical cylinder, Δρ fixed")

In [ ]:
fit_cyl["corr"]

The horizontal position is well determined; the radius, top and bottom
are uncertain by up to a few metres, and correlated at between ±0.8 and
±0.97. A fatter cylinder
needs to be shorter, or deeper, to give the same anomaly. (The fit uses
12-point quadrature to keep it quick; that is accurate to a small
fraction of a microgal here.)

### 6.8 How well determined is each parameter? Repeat with new noise

The $\pm1\sigma$ values above come from a linearisation about the
answer. A direct check: draw fresh noise many times, fit each time, and
look at the spread of the answers. That is what a hundred repeat surveys
would give.

The cell below does it for the two bodies with the most poorly determined
parameters. It takes about ten seconds.

In [ ]:
N_REPEAT = 200
t0 = time.time()
rng_mc = np.random.default_rng(SEED + 1)
mc_prism, mc_cyl = [], []
for _ in range(N_REPEAT):
    d = clean["2-D prism"] + rng_mc.normal(0, NOISE, X_PROFILE.size)
    sol = least_squares(lambda p: (prism_2d(X_PROFILE, *p) - d) / NOISE, fit_prism["p"],
                        bounds=([-50, 0.1, 0, 0.1], [50, 60, 40, 40]), x_scale="jac")
    mc_prism.append(sol.x)
    d = clean["vertical cylinder"] + rng_mc.normal(0, NOISE, X_MAP.size)
    sol = least_squares(lambda p: (cylinder_3d(X_MAP, Y_MAP, *p, n=12).ravel() - d) / NOISE,
                        fit_cyl["p"], bounds=([-50, -50, 0.5, 0, 1], [50, 50, 40, 40, 200]),
                        x_scale="jac")
    mc_cyl.append(sol.x)
mc_prism, mc_cyl = np.array(mc_prism), np.array(mc_cyl)
print(f"{2 * N_REPEAT} fits in {time.time() - t0:.0f} s")

lam_mc = mc_prism[:, 1] * mc_prism[:, 3]
print(f"prism:    width {mc_prism[:, 1].mean():.1f} ± {mc_prism[:, 1].std():.1f} m, "
      f"thickness {mc_prism[:, 3].mean():.1f} ± {mc_prism[:, 3].std():.1f} m, "
      f"but w × t = {lam_mc.mean():.1f} ± {lam_mc.std():.1f} m² (true 50)")
vol = np.pi * mc_cyl[:, 2] ** 2 * (mc_cyl[:, 4] - mc_cyl[:, 3])
print(f"cylinder: radius {mc_cyl[:, 2].mean():.1f} ± {mc_cyl[:, 2].std():.1f} m, "
      f"bottom {mc_cyl[:, 4].mean():.1f} ± {mc_cyl[:, 4].std():.1f} m, "
      f"but mass {vol.mean() * DRHO:.0f} ± {vol.std() * DRHO:.0f} t "
      f"(true {np.pi * 36 * (22 - 6.3):.0f})")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 4))
ax = axs[0]
ax.scatter(mc_prism[:, 1], mc_prism[:, 3], s=14, color=COLOR["2-D prism"], alpha=0.7,
           label="one noisy survey each")
w = np.linspace(4, 20, 100)
ax.plot(w, 50.0 / w, color=DATA_COLOR, lw=1, ls="--", label="w × t = 50 m² (true λ)")
ax.plot(PRISM_P["width"], PRISM_P["thickness"], "k+", ms=14, mew=2, label="truth")
ax.set(xlabel="width (m)", ylabel="thickness (m)", title="2-D prism: width against thickness",
       xlim=(4, 20), ylim=(0, 12))
ax.legend(fontsize=8)

ax = axs[1]
ax.scatter(mc_cyl[:, 2], mc_cyl[:, 4], s=14, color=COLOR["vertical cylinder"], alpha=0.7)
ax.plot(CYLINDER["radius"], CYLINDER["bottom"], "k+", ms=14, mew=2)
ax.set(xlabel="radius (m)", ylabel="bottom (m)", title="Vertical cylinder: radius against bottom")

ax = axs[2]
ax.scatter(mc_cyl[:, 2], mc_cyl[:, 3], s=14, color=COLOR["vertical cylinder"], alpha=0.7)
ax.plot(CYLINDER["radius"], CYLINDER["top"], "k+", ms=14, mew=2)
ax.set(xlabel="radius (m)", ylabel="top (m)", title="Vertical cylinder: radius against top")
plt.tight_layout()

Each dot is the answer from one survey. The answers do not scatter in a
round cloud but along a **curve**: for the prism, the curve of constant
$w\,t$. Somewhere along that curve every answer fits its data equally
well, and the noise decides where. The product — the mass — is
well determined even when its factors are not.

### 6.9 Fit the wrong shape

So far we told the inversion the right shape. What if we tell it the wrong
one? Fit every data set with every body of its own dimension, and look at
$\chi^2/N$: a value near 1 means "fits to within the noise".

One more row: fit the **profile through the middle of the sphere's map**
with a **2-D line mass**, as someone with only one line of data and no
reason to doubt that the body was long might do.

In [ ]:
def fit_any(shape, d, x=None):
    """Fit data d with the named shape, starting from the rules of thumb."""
    if shape == "line mass":
        x0, z0, lam0 = guess_2d(x, d)
        return fit(lambda x0, z, l: line_mass_2d(x, x0, z, l * TONNE),
                   ["x0", "depth", "λ (t/m)"], [x0, z0, lam0 / TONNE], d)
    if shape == "2-D prism":
        x0, z0, lam0 = guess_2d(x, d)
        s = np.sqrt(lam0 / (DRHO * KG_M3))
        return fit(lambda *p: prism_2d(x, *p), ["xc", "w", "top", "t"],
                   [x0, s, max(z0 - s / 2, 0.1), s], d,
                   bounds=([-50, 0.1, 0, 0.1], [50, 60, 40, 40]))
    x0, y0, z0, m0 = guess_3d(d)
    if shape == "point mass":
        return fit(lambda x0, y0, z, m: point_mass_3d(X_MAP, Y_MAP, x0, y0, z, m * TONNE).ravel(),
                   ["x0", "y0", "depth", "M (t)"], [x0, y0, z0, m0 / TONNE], d)
    if shape == "sphere":
        R0 = min((3 * m0 / (4 * np.pi * DRHO * KG_M3)) ** (1 / 3), 0.9 * z0)
        return fit(lambda *p: sphere_3d(X_MAP, Y_MAP, *p).ravel(), ["x0", "y0", "depth", "R"],
                   [x0, y0, z0, R0], d, bounds=([-50, -50, 0.5, 0.5], [50, 50, 60, 30]))
    if shape == "vertical cylinder":
        R0 = (m0 / (2 * np.pi * DRHO * KG_M3)) ** (1 / 3)
        return fit(lambda *p: cylinder_3d(X_MAP, Y_MAP, *p, n=12).ravel(),
                   ["x0", "y0", "R", "top", "bottom"], [x0, y0, R0, max(z0 - R0, 0.5), z0 + R0],
                   d, bounds=([-50, -50, 0.5, 0, 1], [50, 50, 40, 40, 200]))


# Which parameter is "depth": to the centre for the line, point and sphere,
# to the top for the prism and the cylinder.
DEPTH_INDEX = {"line mass": 1, "2-D prism": 2, "point mass": 2, "sphere": 2,
               "vertical cylinder": 3}
rows = []
for truth_name, shapes in (("line mass", ["line mass", "2-D prism"]),
                           ("2-D prism", ["line mass", "2-D prism"]),
                           ("point mass", ["point mass", "sphere", "vertical cylinder"]),
                           ("sphere", ["point mass", "sphere", "vertical cylinder"]),
                           ("vertical cylinder", ["point mass", "sphere", "vertical cylinder"])):
    for shape in shapes:
        r = fit_any(shape, data[truth_name], X_PROFILE)
        rows.append({"data from": truth_name, "fitted with": shape, "chi²/N": r["chi2"],
                     "fitted depth (m)": r["p"][DEPTH_INDEX[shape]]})

# One profile across the 3-D sphere, interpreted as 2-D.
row = data["sphere"].reshape(X_MAP.shape)[XY_MAP.size // 2]
r = fit_any("line mass", row, XY_MAP)
rows.append({"data from": "sphere, profile y = 0 only", "fitted with": "line mass",
             "chi²/N": r["chi2"], "fitted depth (m)": r["p"][1]})
pd.DataFrame(rows).set_index(["data from", "fitted with"])

Every shape fits every data set of its own dimension to within the noise.
**The data cannot choose the shape; you have to.** The depths differ,
though — the depth to the top of a prism or cylinder is not the depth to a
point mass — so the choice matters for anything you dig, drill or build.

The last row is the classic mistake: one profile across a round body,
interpreted as 2-D. It fits (at least as well as the right shape), and
puts the body about a fifth too shallow (≈ 10 m instead of 13 m), because the 2-D rule is
$z = x_{1/2}$ where the 3-D one is $z = 1.3\,x_{1/2}$. A second line at
right angles, or a map, would have shown the anomaly was round.

## 7. Gridded inversion

Now assume **nothing** about the shape. Divide the ground into cells,
each with its own unknown density contrast $m_j$. Because gravity is
linear in density, each datum is a weighted sum of the cells,

$$d_i = \sum_j G_{ij}\, m_j \qquad\text{or}\qquad \mathbf d = \mathbf G\,\mathbf m ,$$

where $G_{ij}$ is the anomaly at station $i$ of cell $j$ with unit
density: the 2-D prism formula of section 2.2 for each cell of a
cross-section, or its 3-D equivalent (the prism formula of Plouff, 1976,
and Nagy et al., 2000) for each block of a 3-D mesh.

### 7.1 Many more unknowns than data

There are thousands of cells and tens or hundreds of data, so infinitely
many models fit the data exactly — we have just seen five very different
bodies that do. The inversion has to be told **which** of them we want.
The standard choice (Li and Oldenburg, 1996, 1998) is the **smallest and
smoothest** model that fits the data to within their noise:

$$\min_{\mathbf m}\ \ \phi_d + \beta\,\phi_m, \qquad
\phi_d = \sum_i\left(\frac{(\mathbf G\mathbf m)_i - d_i}{\sigma_i}\right)^2,$$

$$\phi_m = \alpha_s\!\int (w\,m)^2\,dV
        + \int \left|\nabla (w\,m)\right|^2 dV .$$

* **$\beta$** trades fitting the data against simplicity. It is chosen so
  that $\phi_d = N$: every datum fit to about one standard deviation, no
  better (the **discrepancy principle**).
* **$w(z) = (z + z_0)^{-\nu}$ is a depth weight.** Without it, the
  smallest model puts all the mass at the surface: a shallow cell explains
  a datum with a much smaller density than a deep one, because the
  kernel falls off with distance. Weighting by the inverse square root of
  the kernel's decay cancels that: $\nu = 1/2$ in 2-D (kernel $\sim 1/r$)
  and $\nu = 1$ in 3-D (kernel $\sim 1/r^2$).
* The problem is linear and quadratic, so the minimiser has a closed form,
  $\mathbf m = \mathbf B\mathbf G^T(\mathbf G\mathbf B\mathbf G^T + \beta\boldsymbol\Sigma)^{-1}\mathbf d$
  with $\mathbf B = (\mathbf L^T\mathbf L)^{-1}$, $\phi_m = \|\mathbf L\mathbf m\|^2$.

`pf.LinearInversion` does all of this, for a 2-D or a 3-D mesh. The
tunnel notebook (`gravity_magnetics_tunnel.ipynb`) explores the choice of
$\beta$, bounds and compact inversion in more depth; here we use the
defaults.

### 7.2 2-D: one profile, a cross-section of cells

Cells 1 m square, 100 m across and 30 m deep, with padding cells that grow
outwards beyond that.

In [ ]:
mesh2 = pf.make_mesh(-50, 50, 30, cell=1.0)
G2 = pf.gravity_kernel(mesh2, X_PROFILE, HEIGHT)
print(f"2-D mesh: {mesh2.shape[1]} × {mesh2.shape[0]} = {mesh2.n_cells} cells, "
      f"{X_PROFILE.size} data;  G is {G2.shape}")

lin2, inv2 = {}, {}
for name in ("line mass", "2-D prism"):
    lin2[name] = pf.LinearInversion(mesh2, G2, data[name], NOISE, length=3.0, z0=HEIGHT)
    inv2[name] = lin2[name].solve()
inv2_flat = pf.LinearInversion(mesh2, G2, data["line mass"], NOISE, length=3.0,
                               weighting="none").solve()
for name, r in [*inv2.items(), ("line mass, no depth weight", inv2_flat)]:
    print(f"{name:28s} beta = {r.beta:.2e}, chi^2/N = {r.chi2:.2f}, "
          f"largest density {r.model.max():.2f} g/cm³")

In [ ]:
def show_section(ax, mesh, model, title, vmax, outline):
    pc = ax.pcolormesh(mesh.x_edges, mesh.z_edges, model.reshape(mesh.shape), cmap="RdBu_r",
                       vmin=-vmax, vmax=vmax, shading="flat")
    outline(ax)
    ax.set(xlim=(-30, 30), ylim=(30, 0), aspect="equal", title=title,
           xlabel="x (m)", ylabel="depth (m)")
    ax.grid(False)
    return pc


vmax = 0.25
fig, axs = plt.subplots(2, 3, figsize=(15, 6.2), height_ratios=[1.5, 1])
panels = [("line mass", inv2["line mass"], "line-mass data"),
          ("2-D prism", inv2["2-D prism"], "prism data"),
          ("line mass", inv2_flat, "line-mass data, no depth weighting")]
for col, (name, r, title) in enumerate(panels):
    pc = show_section(axs[0, col], mesh2, r.model, title, vmax, draw_bodies_2d)
    ax = axs[1, col]
    ax.errorbar(X_PROFILE, data[name], NOISE, fmt="o", ms=3, color=DATA_COLOR,
                elinewidth=0.8, label="observed")
    ax.plot(X_PROFILE, r.predicted, color=COLOR[name], label=f"predicted, χ²/N = {r.chi2:.2f}")
    ax.set(xlabel="x (m)", ylabel="$g_z$ (mGal)")
    ax.legend(fontsize=8, loc="upper left")
fig.colorbar(pc, ax=axs[0], label="density contrast (g/cm³)", shrink=0.8)

With depth weighting, the line-mass data and the prism data give **almost
the same image** — as they must, since the data are almost the same. The
blob is centred near the right depth, but it is far wider and far weaker
than either body: the largest value is about 0.1 g/cm³ where the truth is
1.0. A smooth inversion spreads the mass out as far as the data allow.

Look also at the **data fit**. $\chi^2/N = 1$, yet the predicted curve
runs *below* the stations over the peak. A blob that wide cannot make a
peak that sharp, so the smooth model spends its misfit budget there. We
come back to this in 7.4.

Without depth weighting (right) the same data are fit just as well by a
thin skin of density just under the stations: the smallest model is the
shallowest one. Nothing in the data rules it out.

### 7.3 3-D: a map, a block of cells

Cells 2.5 m on a side under the 80 × 80 m map, down to 30 m, plus padding
— about 25 600 cells for 441 data. Building $\mathbf G$ and factorising
the regularisation takes ten seconds or so; each inversion after that is
quick.

In [ ]:
t0 = time.time()
mesh3 = pf.make_mesh_3d(-40, 40, -40, 40, 30, cell=2.5)
G3 = pf.gravity_kernel_3d(mesh3, X_MAP, Y_MAP, HEIGHT)
print(f"3-D mesh: {' × '.join(map(str, mesh3.shape[::-1]))} = {mesh3.n_cells} cells; "
      f"G is {G3.shape} ({G3.nbytes / 1e6:.0f} MB), {time.time() - t0:.1f} s")

lin3, inv3 = {}, {}
for name in ("sphere", "vertical cylinder"):
    t0 = time.time()
    lin3[name] = pf.LinearInversion(mesh3, G3, data[name], NOISE, length=5.0, z0=HEIGHT)
    inv3[name] = lin3[name].solve()
    r = inv3[name]
    print(f"{name:18s} beta = {r.beta:.2e}, chi^2/N = {r.chi2:.2f}, "
          f"largest density {r.model.max():.2f} g/cm³  ({time.time() - t0:.1f} s)")

In [ ]:
nz, ny, nx = mesh3.shape
mid = [ny // 2 - 1, ny // 2]                      # the two rows of cells either side of y = 0
k13 = np.argmin(abs(mesh3.z_centers - 13))        # the depth slice nearest 13 m
vmax = 0.25
fig, axs = plt.subplots(2, 3, figsize=(15, 8), width_ratios=[1.25, 1, 1],
                        layout="constrained")
for row, name in enumerate(("sphere", "vertical cylinder")):
    M = inv3[name].model.reshape(mesh3.shape)
    ax = axs[row, 0]
    pc = ax.pcolormesh(mesh3.x_edges, mesh3.z_edges, M[:, mid, :].mean(axis=1), cmap="RdBu_r",
                       vmin=-vmax, vmax=vmax, shading="flat")
    draw_bodies_3d(ax)
    ax.set(xlim=(-30, 30), ylim=(30, 0), aspect="equal", xlabel="x (m)", ylabel="depth (m)",
           title=f"{name} data: section at y = 0")
    ax = axs[row, 1]
    ax.pcolormesh(mesh3.x_edges, mesh3.y_edges, M[k13], cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                  shading="flat")
    for name2, R in (("sphere", SPHERE["radius"]), ("vertical cylinder", CYLINDER["radius"])):
        ax.add_patch(Circle((0, 0), R, fill=False, ec=COLOR[name2], lw=1.5))
    ax.set(xlim=(-30, 30), ylim=(-30, 30), aspect="equal", xlabel="x (m)", ylabel="y (m)",
           title=f"depth slice at {mesh3.z_centers[k13]:.1f} m")
    ax = axs[row, 2]
    res = ((data[name] - inv3[name].predicted) / NOISE).reshape(X_MAP.shape)
    pr = ax.pcolormesh(X_MAP, Y_MAP, res, cmap="RdBu_r", vmin=-3, vmax=3, shading="nearest")
    ax.set(aspect="equal", xlabel="x (m)", ylabel="y (m)",
           title=f"residual / σ,  χ²/N = {inv3[name].chi2:.2f}")
for ax in axs.ravel():
    ax.grid(False)
fig.colorbar(pc, ax=axs[:, :2], label="density contrast (g/cm³)", shrink=0.5)
fig.colorbar(pr, ax=axs[:, 2], label="(observed − predicted) / σ", shrink=0.5)

The same story in 3-D. The sphere and the cylinder give indistinguishable
images: a round blob centred under the anomaly, densest at about the right
depth, much weaker and larger than the truth, and smeared **downward**
— the inversion knows the body is somewhere below, but not how far its
mass extends.

The residual maps are noise everywhere **except over the peak**, where a
patch of stations is under-predicted by two or three standard deviations.

### 7.4 Read the residuals, not just χ²

The discrepancy principle asks for $\chi^2 = N$ **in total**. It does not
ask that the misfit be spread evenly, and the smoothest model puts it
where it is cheapest for a smooth model: on the few stations over the
peak. A residual that is the same sign at neighbouring stations is
**signal the model has not explained**, whatever $\chi^2$ says.

How much smaller does $\beta$ have to be before the peak is fit? The table
gives the mean normalised residual on the stations over the peak (within
4 m of it in 2-D, 6 m in 3-D) as $\beta$ falls below the discrepancy
value. Each $\beta$ is a quick solve, since the expensive set-up is done.

In [ ]:
rows = []
for label, lin, x_dist, dname in (
        ("2-D, line mass", lin2["line mass"], np.abs(X_PROFILE), "line mass"),
        ("3-D, sphere", lin3["sphere"], np.hypot(X_MAP, Y_MAP).ravel(), "sphere")):
    peak = x_dist <= (4 if label.startswith("2-D") else 6)
    beta0 = lin.target_beta()
    for factor in (1, 0.3, 0.1, 0.03):
        r = lin.solve(beta=beta0 * factor)
        res = (data[dname] - r.predicted) / NOISE
        rows.append({"": label, "β / β(χ²=N)": factor, "χ²/N": r.chi2,
                     "mean residual over peak (σ)": res[peak].mean(),
                     "largest density (g/cm³)": r.model.max(),
                     "most negative (g/cm³)": r.model.min()})
pd.DataFrame(rows).set_index(["", "β / β(χ²=N)"]).round(2)

In 2-D, fitting the peak takes a $\beta$ about ten times smaller, and by
then the model is fitting the noise elsewhere ($\chi^2/N \approx 0.5$) and
growing negative side-lobes that are not in the ground. In 3-D even that is
not enough. Neither $\beta$ is "right": the smooth model is simply the
wrong *kind* of model for a compact body, and no $\beta$ makes it the
right kind. Two honest responses are to report the smooth model together
with its residual map, or to change the regularisation — the compact
(minimum-support) inversion of the tunnel notebook
(`LinearInversion.solve_compact`) asks for a body with sharp edges, and
fits the peak better, though not perfectly.

### 7.5 What the data *do* determine

Both kinds of inversion agree on three things, and these are the things
gravity data really constrain:

1. **Horizontal position.** Exact in every fit.
2. **Depth, roughly.** The parametric fits give the depth to the centre of
   mass well when told the shape; the smooth image puts its peak close to it.
3. **Excess mass.** By Gauss's theorem the anomaly integrated over the
   whole surface is fixed by the mass alone, whatever its shape:
   $\int g_z\,dx = 2\pi G\lambda$ in 2-D and $\iint g_z\,dx\,dy = 2\pi G M$
   in 3-D. A survey that covered the anomaly completely would measure
   the mass with no assumption at all.

The cell below compares three estimates of the mass: Gauss's theorem
applied straight to the data, the sum $\sum_j m_j V_j$ over the gridded
model, and the parametric fit.

In [ ]:
rows = []
for label, lin, x_dist, dname in (
        ("2-D, line mass", lin2["line mass"], np.abs(X_PROFILE), "line mass"),
        ("3-D, sphere", lin3["sphere"], np.hypot(X_MAP, Y_MAP).ravel(), "sphere")):
    peak = x_dist <= (4 if label.startswith("2-D") else 6)
    beta0 = lin.target_beta()
    for factor in (1, 0.3, 0.1, 0.03):
        r = lin.solve(beta=beta0 * factor)
        res = (data[dname] - r.predicted) / NOISE
        rows.append({"": label, "β / β(χ²=N)": factor, "χ²/N": r.chi2,
                     "mean residual over peak (σ)": res[peak].mean(),
                     "largest density (g/cm³)": r.model.max(),
                     "most negative (g/cm³)": r.model.min()})
vol2 = mesh2.area                                         # m^2 per metre along strike
vol3 = mesh3.volume
rows = []
for name, lam_true in (("line mass", LINE["mass_per_length"]), ("2-D prism", lam_prism)):
    rows.append({"data": name, "unit": "t/m", "true": lam_true / TONNE,
                 "Gauss, from the data": trapezoid(data[name], X_PROFILE) / MGAL
                 / (2 * np.pi * G) / TONNE,
                 "gridded model": np.sum(inv2[name].model * vol2) * KG_M3 / TONNE,
                 "parametric fit": (fit_line["p"][2] if name == "line mass"
                                    else fit_prism["p"][1] * fit_prism["p"][3] * DRHO
                                    * KG_M3 / TONNE)})
m_cyl_true = np.pi * CYLINDER["radius"] ** 2 * (CYLINDER["bottom"] - CYLINDER["top"]) * DRHO * KG_M3
for name, m_true, m_fit in (
        ("sphere", POINT["mass"], 4 / 3 * np.pi * fit_sphere["p"][3] ** 3 * DRHO * KG_M3),
        ("vertical cylinder", m_cyl_true,
         np.pi * fit_cyl["p"][2] ** 2 * (fit_cyl["p"][4] - fit_cyl["p"][3]) * DRHO * KG_M3)):
    d_map = data[name].reshape(X_MAP.shape)
    rows.append({"data": name, "unit": "t", "true": m_true / TONNE,
                 "Gauss, from the data": trapezoid(trapezoid(d_map, XY_MAP), XY_MAP)
                 / MGAL / (2 * np.pi * G) / TONNE,
                 "gridded model": np.sum(inv3[name].model * vol3) * KG_M3 / TONNE,
                 "parametric fit": m_fit / TONNE})
pd.DataFrame(rows).set_index(["data", "unit"]).round(1)

* **Gauss from the data** comes up short, by about 15 % in 2-D and 25–30 %
  in 3-D: the survey stops at ±40 m while the anomaly's tails carry on.
  The deeper the body, the wider the tails and the bigger the survey
  needed. It is still the only estimate here that assumes nothing.
* **The gridded model** fills in the missing tails with mass in the
  padding cells beyond the survey and down to the bottom of the mesh. It
  has to *guess* how much: here it overshoots by about 20 % in 2-D, and in
  3-D lands 10 % low for the sphere and 30 % high for the cylinder, from
  data that differ by less than the noise. Change the mesh or the
  regularisation and the guess changes.
* **The parametric fit** is closest, because it was told the shape. If
  the shape is right, that is the best estimate; if it is wrong (section
  6.9), the mass is usually still close, while the size is not.

## 8. Summary

| | What it needs from you | What it gives |
|---|---|---|
| Rules of thumb (§2–3) | 2-D or 3-D | depth to centre, mass |
| Parametric fit (§6) | the shape, and $\Delta\rho$ | sharp numbers with error bars — for that shape |
| Gridded inversion (§7) | a mesh, a regularisation, $\beta$ | a smooth image; position and approximate depth, never the true size or contrast |

The five bodies in this notebook make anomalies that differ by less than
the noise of any gravimeter. That is not a quirk of the example: it is the
basic non-uniqueness of potential fields. Gravity data determine **where**
a mass excess is and **how much** of it there is, and constrain its depth;
the **shape** and the **density** have to come from geology, drilling or
another geophysical method.

### Things to try

* Make the bodies **voids**: set `DRHO = -1.0`. Every curve turns over;
  nothing else changes.
* Move the 3-D bodies **deeper** (`depth` 20 m, `top` and `bottom` +7 m).
  The anomaly widens and weakens, the parametric uncertainties grow, and
  Gauss's theorem misses more of the mass on the same survey.
* Raise `NOISE` to 0.02 mGal (a careless survey, or a windy day) and
  re-run sections 5–7. Which parameters survive?
* Shrink the map to ±20 m. Which estimates of the mass suffer?
* In section 6.7 start the cylinder fit from a very tall, thin cylinder.
  Does it find the same answer?
* In section 7, change `length` (the smoothing length) or `z0` and see how
  the depth of the blob moves.

### References

Blakely, R. J. (1995). *Potential Theory in Gravity and Magnetic
Applications.* Cambridge University Press.

Li, Y., and Oldenburg, D. W. (1998). 3-D inversion of gravity data.
*Geophysics*, 63(1), 109–119.

Nagy, D., Papp, G., and Benedek, J. (2000). The gravitational potential and
its derivatives for the prism. *Journal of Geodesy*, 74, 552–560.

Plouff, D. (1976). Gravity and magnetic fields of polygonal prisms and
application to magnetic terrain corrections. *Geophysics*, 41(4), 727–741.

Telford, W. M., Geldart, L. P., and Sheriff, R. E. (1990). *Applied
Geophysics*, 2nd ed. Cambridge University Press.